# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** OpenCode exited with status 0 without a valid queue-result.json. Last captured output: [2026-10-09T10:53:27.078Z] ! permission requested: external_directory (/proc/2074691/*); auto-rejecting | [2026-10-09T10:53:27.086Z] ✗ cat /proc/2074691/cmdline 2>/dev/null | tr '\0' ' '; echo; ps -o pid,ppid,etime,stat,cmd -p 2074691 2>/dev/null; pgrep -fa "colab_session_guard"  failed | [2026-10-09T10:53:27.086Z] Error: The user rejected permission to use this specific tool call. | Required queue-result.json is unavailable or invalid: FileNotFoundError

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: Integrating Crop Growth Models and Genomic Prediction to Improve Flowering Time Forecasting Across Novel Environments and Breeding Lines

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-91bf3367cf5522380c1e30eebab78cdb)


# CGM-WGP × RN-GBLUP — Flowering-Time Forecasting in Common Bean (CV2 Sparse Testing)

**Paper:** *Integrating Crop Growth Models and Genomic Prediction to Improve Flowering Time
Forecasting Across Novel Environments and Breeding Lines* — Research Square preprint v1,
DOI [10.21203/rs.3.rs-9890445/v1](https://doi.org/10.21203/rs.3.rs-9890445/v1) (posted 2026-07-14).

**Author code:** [`melycg2001/CGM-WGP`](https://github.com/melycg2001/CGM-WGP) pinned to commit
`62d0dd8f6cf67de5857149cfb1a2b1a8ef2dc4ab` (the repository's single commit; prototype, no tagged release).

**Executed scope (partial demonstration):** one crop — common bean (*Phaseolus vulgaris*) — under
**one** cross-validation scheme, `balanced_training_subset` (G×E sparse testing, CV2): an 80/20
per-environment cell split repeated 5 times with seed 42. The author pipeline fits **four** methods
on identical folds — Mechanistic (cardinal-β + photoperiod), GBLUP, RN-GBLUP (Jarquín 2014 reaction
norm), and **CGM-WGP** (EM + L-BFGS-B joint fit of per-genotype thermal-time Θ, photoperiod rate
penalty S, and critical daylength P_c under a genomic prior) — and compares predicted vs observed
flowering DAP.

**Not reproduced:** broccoli, the other three CV schemes (`new_varieties`, `geographic`,
`cv00_double_novelty`), full-text-level benchmark tables, and any dataset-level accuracy claim.

**日本語要約:** このノートブックは、Research Square プレプリント v1 の CGM-WGP
（作物成長モデル＋全ゲノム予測）と RN-GBLUP の比較を、著者公開コード
(コミット `62d0dd8`) をそのまま使って **インゲンマメ 1 作物 × CV2 スパース
テスティング 1 方式** で再現するものです。全 4 手法を同一分割で学習し、
開花日 (DAP) の予測値と観測値を比較します。ブロッコリー・他の CV 方式・
論文レベルのベンチマーク表は対象外です。**この論文のコードは純 Python
(numpy/scipy) なので GPU は不要で、CPU ランタイムで実行します。**

## Runtime selection — CPU only

**Select `CPU` runtime in Colab (Runtime → Change runtime type → CPU).** The pinned
pipeline is pure numpy/pandas/scipy optimization plus one R subprocess; it contains no
GPU/CUDA code path, so a GPU runtime would add nothing faithful.

- Run all: `Runtime → Run all`.
- Setup (R apt package + pip) ≈ 2–4 min; scientific run time is measured in the cells below.
- Downloads: < 1 MB total (author code + small CSVs + one LFS genomic eigen-decomposition).
- **日本語:** GPU は不要です。Colab で「ランタイム → ランタイムのタイプを変更 → CPU」を
  選択してから「ランタイム → すべてのセルを実行」してください。セットアップ数分 +
  解析 (所要時間は下のセルで実測表示されます)。ダウンロード合計 1 MB 未満。

In [ ]:
import platform, multiprocessing, os
print("Python:", platform.python_version())
print("Platform:", platform.platform())
print("CPU cores:", multiprocessing.cpu_count())
try:
    print("Free RAM (GB):", round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_AVPHYS_PAGES') / 1e9, 1))
except (ValueError, AttributeError, OSError):
    print("Free RAM: unavailable on this platform")

## 1 · Environment and dependencies

The author's `requirements.txt` lists numpy, pandas, scipy, pyyaml, openpyxl, matplotlib,
plotly, and requests. Only **numpy / pandas / scipy / matplotlib / pyyaml** are imported by
the CV-run path we execute (`cgm_wgp.app` → `main_fit`, `mech_fit`, `joint_fit`,
`jarquin_loo`, `gblup`); openpyxl and plotly belong to data-prep and diagnostic scripts we
do not invoke.

**R is required**: the author's `load_evd()` (`src/cgm_wgp/gblup.py`) reads the pre-computed
genomic eigen-decomposition `pipeline/2_gmatrix/Beans/EVD.rda` by shelling out to
`Rscript` with a short base-R `load()`/`write.csv` script — no external R packages (AGHmatrix
is only needed to *generate* an EVD, which we skip by using the author's shipped one).

**日本語:** 著者の `requirements.txt` のうち、実行経路が使うのは numpy / pandas /
scipy / matplotlib / pyyaml のみです。加えて、著者の `load_evd()` が EVD.rda を
Rscript サブプロセスで読むため **R 本体 (r-base-core)** をインストールします
（外部 R パッケージは不要）。

In [ ]:
import subprocess, sys, shutil
# Pip: Colab already ships recent numpy/pandas/scipy/matplotlib; this only tops up
# missing pieces and pyyaml (no-op when satisfied, default upgrade strategy).
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy", "pandas", "scipy", "matplotlib", "pyyaml"], check=True)
# R: needed only for the author's Rscript-based EVD loader (gblup.load_evd).
if not shutil.which("Rscript"):
    subprocess.run(["sudo", "apt-get", "-qq", "update"], check=True)
    subprocess.run(["sudo", "apt-get", "-qq", "install", "-y", "r-base-core"], check=True)
r = subprocess.run(["Rscript", "-e", "cat(R.version.string)"], capture_output=True, text=True, check=True)
print("R:", r.stdout.strip())
import numpy, pandas, scipy, matplotlib, yaml
print("numpy", numpy.__version__, "| pandas", pandas.__version__,
      "| scipy", scipy.__version__, "| matplotlib", matplotlib.__version__)

## 2 · Acquire the pinned author code and minimal bean inputs

We fetch exactly the paths this demonstration needs from the pinned commit using a
**partial clone with sparse checkout** (blobless; only selected blobs are downloaded) and
**`GIT_LFS_SKIP_SMUDGE=1`** so the Git-LFS pointer `pipeline/2_gmatrix/Beans/EVD.rda`
is not auto-smudged; its real payload is pulled explicitly afterwards. The checkout is
asserted to be exactly at `62d0dd8f6cf67de5857149cfb1a2b1a8ef2dc4ab`.

Files retrieved (all < 1 MB together):
- `src/cgm_wgp/` — the author's Python package; `configs/beans.yaml` — crop config
  (cardinal temperatures Tb=10/Topt=25/Tc=38 °C, CV2 settings, joint-fit bounds).
- `pipeline/1_data/Beans/phenotypes_dated.csv` — pipeline-ready flowering observations
  (the paper's raw source, `3. RIJC Data.xlsx`, is only needed by the separate data-prep
  step, which the app does not run here — see `app.py` defaults).
- `pipeline/1_data/weather/{CIT,ND,PAL,POP,PR}/weather.csv` — daily weather per site.
- `pipeline/2_gmatrix/Beans/EVD.rda` — genomic eigen-decomposition (LFS payload).

**日本語:** ピン留めしたコミットから部分クローン (blobless + sparse checkout) で
必要なファイルだけを取得します。`EVD.rda` は Git LFS で管理されているため
`GIT_LFS_SKIP_SMUDGE=1` でポインターのまま取得し、後から実体のみを明示的に
pull します。チェックアウトが `62d0dd8...` と一致することを assert で確認します。

In [ ]:
import os, shutil, subprocess, hashlib
from pathlib import Path

REPO_URL = "https://github.com/melycg2001/CGM-WGP.git"
PINNED_SHA = "62d0dd8f6cf67de5857149cfb1a2b1a8ef2dc4ab"
REPO_DIR = Path("/content/CGM-WGP")

def run(cmd, cwd=None, env=None):
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, env=env, check=True)

env = dict(os.environ, GIT_LFS_SKIP_SMUDGE="1")
if not (REPO_DIR / ".git").exists():
    run(["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(REPO_DIR)], env=env)
    # Sparse patterns: author code + config + the bean pipeline inputs only
    # (skips the LFS raw-source xlsx/docx and the broccoli dataset).
    run(["git", "sparse-checkout", "set", "--no-cone",
         "/src/**", "/configs/**", "/README.md", "/requirements.txt",
         "/pipeline/1_data/Beans/phenotypes_dated.csv",
         "/pipeline/1_data/weather/**",
         "/pipeline/2_gmatrix/Beans/*.rda"], cwd=REPO_DIR)
    run(["git", "checkout", PINNED_SHA], cwd=REPO_DIR, env=env)

head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR,
                      capture_output=True, text=True, check=True).stdout.strip()
assert head == PINNED_SHA, f"checkout is {head}, expected {PINNED_SHA}"
print("checkout OK:", head)

# Real LFS payload for the bean EVD (pointer file is ~131 bytes; the payload is not).
evd = REPO_DIR / "pipeline/2_gmatrix/Beans/EVD.rda"
if evd.stat().st_size < 300 or evd.read_bytes().startswith(b"version https://git-lfs"):
    if not shutil.which("git-lfs"):
        subprocess.run(["sudo", "apt-get", "-qq", "install", "-y", "git-lfs"], check=True)
        run(["git", "lfs", "install"], cwd=REPO_DIR)
    try:
        run(["git", "lfs", "pull", "--include", "pipeline/2_gmatrix/Beans/EVD.rda"], cwd=REPO_DIR)
    except subprocess.CalledProcessError:
        # Documented fallback: resolve the pointer via GitHub's media endpoint.
        import urllib.request
        url = f"https://media.githubusercontent.com/media/melycg2001/CGM-WGP/{PINNED_SHA}/pipeline/2_gmatrix/Beans/EVD.rda"
        print("$ GET", url)
        urllib.request.urlretrieve(url, evd)

data = evd.read_bytes()
assert not data.startswith(b"version https://git-lfs"), "EVD.rda is still an LFS pointer"
assert len(data) > 300, f"EVD.rda payload too small: {len(data)} bytes"
print(f"EVD.rda: {len(data)} bytes, sha256 {hashlib.sha256(data).hexdigest()[:16]}…")

for rel in ["src/cgm_wgp/app.py", "configs/beans.yaml",
            "pipeline/1_data/Beans/phenotypes_dated.csv",
            "pipeline/1_data/weather/CIT/weather.csv",
            "pipeline/1_data/weather/ND/weather.csv",
            "pipeline/1_data/weather/PAL/weather.csv",
            "pipeline/1_data/weather/POP/weather.csv",
            "pipeline/1_data/weather/PR/weather.csv"]:
    p = REPO_DIR / rel
    assert p.is_file(), f"missing pinned input: {rel}"
    print(f"{p.stat().st_size:>8} bytes  {rel}")

## 3 · Input preview — bean flowering observations

Per the author's `documentation/DATA_FORMAT.md`, `phenotypes_dated.csv` has columns
`id` (genotype), `Planting` (environment: site code), `ft` (flowering time in days after
planting), and `Start_Date`/`End_Date` (observation window). Below we show the actual
shapes, per-environment observation counts, genotype totals, and the flowering-time (DAP)
distribution actually downloaded — this is observed content, not an assumption.

**日本語:** `phenotypes_dated.csv` の実際の列構成・遺伝子型数・環境 (Planting) ごとの
観測数・開花 DAP の分布を表示します。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pheno = pd.read_csv(REPO_DIR / "pipeline/1_data/Beans/phenotypes_dated.csv")
print("shape:", pheno.shape, "| columns:", list(pheno.columns))
print("unique genotypes:", pheno["id"].nunique())
print("\nobservations per planting:")
print(pheno["Planting"].value_counts().sort_index().to_string())
print("\nft (days after planting) summary by planting:")
print(pheno.groupby("Planting")["ft"].describe().round(1).to_string())

fig, ax = plt.subplots(figsize=(8, 3.2))
for planting, grp in pheno.groupby("Planting"):
    ax.hist(grp["ft"].dropna(), bins=25, alpha=0.5, label=str(planting))
ax.set_xlabel("Observed flowering time ft (days after planting)")
ax.set_ylabel("Observations")
ax.set_title("Bean phenotypes_dated.csv @ commit " + PINNED_SHA[:8] + " (input data, per-planting DAP distributions)")
ax.legend(title="Planting (site)")
fig.tight_layout()
plt.show()

## 3b · Input preview — daily weather per site

The mechanistic and photoperiod models drive development with daily `Tmax`/`Tmin`
(configured via `weather.tmax_col`/`tmin_col`, `year_doy` date format). The five bean
sites span latitudes 10.0°N (CIT) to 46.9°N (ND) per `configs/beans.yaml`, which is what
makes the photoperiod contrast identifiable. We verify each site's CSV actually covers the
phenotype observation windows — if a site file were missing or short, the pipeline would
fail loudly later, but we check here first.

**日本語:** 5 サイトの daily weather (`Tmax`/`Tmin`) の行数・期間・温度範囲を確認します。
緯度は 10.0°N〜46.9°N で、これが光周期応答の識別を可能にします。

In [ ]:
import yaml as _yaml
weather_root = REPO_DIR / "pipeline/1_data/weather"
cfg = _yaml.safe_load((REPO_DIR / "configs/beans.yaml").read_text())
rows = []
for site in sorted(pheno["Planting"].astype(str).unique()):
    wf = weather_root / site / "weather.csv"
    assert wf.is_file(), f"missing weather for {site}"
    w = pd.read_csv(wf)
    rows.append({
        "site": site,
        "rows": len(w),
        "date_min": int(w[cfg["weather"]["year_col"]].min()),
        "date_max": int(w[cfg["weather"]["year_col"]].max()),
        "Tmin_min (°C)": round(w[cfg["weather"]["tmin_col"]].min(), 1),
        "Tmax_max (°C)": round(w[cfg["weather"]["tmax_col"]].max(), 1),
        "lat (config)": cfg["locations"][site]["latitude"],
    })
wx = pd.DataFrame(rows)
print(wx.to_string(index=False))

## 4 · Run the author pipeline — CGM-WGP + RN-GBLUP on bean, CV2

This executes the author's documented entry point verbatim
(README "Running both workflows" / QUICKSTART "First Run"):

```
python -m cgm_wgp.app --config configs/beans.yaml --run-joint --cv-scheme balanced_training_subset
```

What it does (`src/cgm_wgp/app.py::_run_cv_scheme`): for each of 5 repetitions (seed 42) it
splits each site's genotype×planting cells 80/20, writes per-fold train/test CSVs, then runs
`cgm_wgp.main_fit` per fold — mechanistic cardinal-β fit via `scipy.optimize.dual_annealing`,
GBLUP and RN-GBLUP via the R-loaded EVD, and (`--run-joint`) the CGM-WGP EM + L-BFGS-B joint
fit — then aggregates predictions, writes `combined/diagnostics/` metrics
(`cv_summary.csv`, `rep_summary.csv` with mean±std across reps) and `plots/obs_vs_pred_*.png`.

**`--maxiter` choice (documented adaptation):** the production default is 2000 dual-annealing
iterations per genotype per repetition. The author's own QUICKSTART uses `--maxiter 30` for the
first full pipeline run; we use a modest value to keep the notebook inside Colab's session
budget. Lowering `maxiter` trades optimizer convergence for wall time — treat the numbers as a
demonstration of the *pipeline mechanics*, not as converged paper-grade estimates.

**日本語:** 著者ドキュメントどおりのコマンドで CV2（各サイトの遺伝子型×環境セルを
80/20 分割 × 5 反復、seed 42）を実行し、4 手法を同一フォールドで学習します。
`--maxiter` は dual_annealing の反復上限で、本ノートブックでは実行時間の制約から
著者 QUICKSTART 相当の小さい値に設定しています（収束度は実運用より低い）。
結果はパイプラインの仕組み確認のためのもので、収束した推定値ではありません。

In [ ]:
import subprocess, sys, time, os
from pathlib import Path

MAXITER = 30  # documented adaptation; production default is 2000 (see markdown above)

env = dict(os.environ, PYTHONPATH="/content/CGM-WGP/src")
cmd = [sys.executable, "-m", "cgm_wgp.app",
       "--config", "configs/beans.yaml",
       "--run-joint",
       "--cv-scheme", "balanced_training_subset",
       "--maxiter", str(MAXITER)]
print("$ PYTHONPATH=/content/CGM-WGP/src " + " ".join(cmd) + "\n")

t0 = time.time()
proc = subprocess.Popen(cmd, cwd=str(REPO_DIR), env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
elapsed = time.time() - t0
assert proc.returncode == 0, f"pipeline exited {proc.returncode}"
print(f"\n=== pipeline finished in {elapsed/60:.1f} min (exit {proc.returncode}) ===")

runs_root = REPO_DIR / "pipeline/output/runs/Beans"
cv_dirs = [d for d in runs_root.rglob("*") if d.is_dir() and (d / "combined").is_dir()]
assert cv_dirs, "no CV run directory produced"
RUN_DIR = max(cv_dirs, key=lambda d: d.stat().st_mtime)
print("run dir:", RUN_DIR)

## 5 · Validation comparison — four methods side by side

The combined step aggregates every repetition's `diagnostics/validation_comparison.csv` into
`per_rep_validation.csv` (one row per rep × method) and `rep_summary.csv` (per-method mean ±
std of MAE, RMSE, bias, and within-N-day hit rates over all held-out genotype×environment
cells). `cv_summary.csv` additionally holds overall and per-environment RMSE and Pearson r.
All values below come from this executed run only.

**日本語:** 各反復の検証指標を集計した `per_rep_validation.csv`（反復×手法ごと）と、
手法ごとの mean±std まとめ `rep_summary.csv`、さらに環境別の RMSE・Pearson r を含む
`cv_summary.csv` を表示します。値はすべてこの実行によるものです。

In [ ]:
import pandas as pd
from IPython.display import display

per_rep = pd.read_csv(RUN_DIR / "combined/diagnostics/per_rep_validation.csv")
summary = pd.read_csv(RUN_DIR / "combined/diagnostics/rep_summary.csv")
cv_summary = pd.read_csv(RUN_DIR / "combined/cv_summary.csv")

print("=== Per-method mean ± std across the 5 CV2 repetitions (all held-out cells) ===")
display(summary)

print("\n=== cv_summary.csv (overall + per-environment RMSE / Pearson r) ===")
display(cv_summary)

print("\n=== Per-repetition validation comparison (raw) ===")
display(per_rep)

## 6 · Predicted vs observed flowering DAP — author-generated scatter plots

The pipeline saved one `obs_vs_pred_<method>.png` per method in the run's `plots/`
directory (`src/cgm_wgp/app.py::_plot_obs_vs_pred`: observed vs predicted DAP colored by
planting, with the Pearson r annotated). We display the saved PNGs directly — these are the
pipeline's own outputs from this executed run, not re-plots.

**日本語:** パイプラインが保存した `obs_vs_pred_*.png`（観測 DAP vs 予測 DAP、
環境別に色分け、Pearson r 付き）をそのまま表示します。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

plots_dir = RUN_DIR / "plots"
saved = sorted(plots_dir.glob("obs_vs_pred_*.png"))
assert saved, "no obs_vs_pred plots were generated"
print("saved plots:", [p.name for p in saved])
ncol = 2
nrow = (len(saved) + ncol - 1) // ncol
fig, axes = plt.subplots(nrow, ncol, figsize=(7 * ncol, 5.4 * nrow))
axes = [axes] if nrow * ncol == 1 else list(axes.flat)
for ax in axes:
    ax.axis("off")
for ax, png in zip(axes, saved):
    ax.imshow(mpimg.imread(str(png)))
    ax.set_title(png.name, fontsize=9)
fig.tight_layout()
plt.show()

## 7 · Interpretation — against the paper's claims

The paper's abstract states that (a) **RN-GBLUP was most accurate under sparse testing**
(training data cover all target environments — exactly this CV2 scheme), and (b) **CGM-WGP
wins for untested environments and G×E combinations** (bean: r = 0.86, RMSE = 5.2 d, from the
untested-environment settings, *not* this CV2 scheme).

Read the `rep_summary.csv` / `cv_summary.csv` tables above with that framing: in this CV2 run
the expectation from the paper is RN-GBLUP ≥ CGM-WGP, while the paper's headline bean numbers
come from a different, harder scenario we deliberately did not run. **A one-crop, one-scheme,
5-repetition demonstration validates the pipeline mechanics — not the paper's dataset-level
conclusions.**

Also visible in `plots/`: `alpha_beta_distribution.png` (per-genotype cardinal-β parameters)
and the joint-fit per-genotype parameters in `combined/params/joint_params.csv`
(Θ thermal time, S photoperiod rate penalty, P_c critical daylength).

**日本語:** 論文アブストラクトでは、(a) スパーステスティング（この CV2）では
RN-GBLUP が最も精度が高い、(b) 未経験環境・G×E では CGM-WGP が優れる
（マメ: r=0.86, RMSE=5.2 d — これは本ノートブックが実行していない別設定）と
述べられています。上の表はその枠組みで読んでください。この実行は
パイプラインの動作検証であり、論文のデータセットレベルの結論の検証ではありません。

## 8 · Sources, validation record, and limitations

**Sources (all retrieved in this notebook's runtime):**
- Paper: Research Square preprint v1, DOI 10.21203/rs.3.rs-9890445/v1.
- Code: `melycg2001/CGM-WGP` @ `62d0dd8f6cf67de5857149cfb1a2b1a8ef2dc4ab` (author's single commit; no license file — "License to be determined"; prototype).
- Data: bean phenotypes + per-site weather + bean genomic EVD, all from the same pinned commit.

**Adaptations (documented, behavior-preserving):** `PYTHONPATH=src` because the prototype
ships no package installer; base-R via apt for the author's `load_evd()` Rscript subprocess;
explicit LFS payload pull for `EVD.rda`; `--maxiter` reduced from the production default
2000 (see the run cell's note).

**Limitations:** single crop, single CV scheme, reduced optimizer budget; the full text of
the preprint was not machine-readable, so paper-level CV2 bean reference numbers are cited
only qualitatively from the abstract. One executed example ≠ dataset-level reproduction.

**日本語:** 使用したのは論文 v1・著者コード `62d0dd8`・同コミットのマメデータの
みです。適応点（PYTHONPATH、R の apt 導入、LFS 明示 pull、maxiter 縮小）と
限界（1 作物・1 CV 方式・低反復・全文未読による定性的比較のみ）は本文の
とおりです。実行 1 例はデータセットレベルの再現ではありません。